# Demo 07f — Tuning Hyperparameters

**Class 07 · E6** (slides: `slides/07_ensembles.md`)

A **parameter** is learned by `fit` (the splits of a tree). A **hyperparameter** is chosen before `fit` (the depth of a tree, the learning rate of boosting). Ensembles have many of them, so a comparison of untuned models says little.

* Three ways to search: a grid, random points, and Optuna's TPE, which learns from earlier trials.
* Pruning: stop a bad trial early.
* Which hyperparameter matters most.
* Nested cross-validation: the best validation score is optimistic.
* Default against tuned models on real tables.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed

from faa import data, demo, metrics, plotlib
from faa.demo import show, slide
from faa.plotlib import BLUE, GREY, ORANGE, RED

demo.setup("07")
SEED = 0
N_JOBS = 8

In [ ]:
import time  # noqa: E402

import optuna  # noqa: E402
from sklearn.base import clone  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split  # noqa: E402
from sklearn.neighbors import KNeighborsClassifier  # noqa: E402
from sklearn.pipeline import make_pipeline  # noqa: E402
from sklearn.preprocessing import StandardScaler  # noqa: E402
from sklearn.svm import SVC  # noqa: E402
from threadpoolctl import threadpool_limits  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402

optuna.logging.set_verbosity(optuna.logging.WARNING)
X_all, y_all = data.load_spam()
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, train_size=1500, stratify=y_all, random_state=SEED)
CV = StratifiedKFold(3, shuffle=True, random_state=SEED)
print(f"spambase: {len(y_tr)} rows to tune on, {len(y_te)} test rows")


def svm_score(log_c, log_gamma, seed=SEED):
    """3-fold CV MCC of an RBF SVM; C and gamma are searched on a log scale."""
    model = make_pipeline(StandardScaler(), SVC(C=10**log_c, gamma=10**log_gamma))
    return float(
        cross_val_score(
            model, X_tr, y_tr, cv=StratifiedKFold(3, shuffle=True, random_state=seed), scoring="matthews_corrcoef"
        ).mean()
    )

## Grid, random or TPE

Each method gets 25 trials on the same two hyperparameters of an SVM. **Grid:** 5 values each, every combination. **Random:** 25 points drawn at random. **TPE** (tree-structured Parzen estimator): splits past trials into good and bad and proposes points that look like the good ones.

In [ ]:
slide("Three Ways to Search")
TRIALS = 25
C_RANGE, GAMMA_RANGE = (-2.0, 3.0), (-5.0, 0.0)


def grid_points():
    return [(c, g) for c in np.linspace(*C_RANGE, 5) for g in np.linspace(*GAMMA_RANGE, 5)]


def random_points(seed):
    rng = np.random.default_rng(seed)
    return list(zip(rng.uniform(*C_RANGE, TRIALS), rng.uniform(*GAMMA_RANGE, TRIALS)))


def tpe_points(seed):
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    with threadpool_limits(1):
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(
            lambda t: svm_score(t.suggest_float("c", *C_RANGE), t.suggest_float("g", *GAMMA_RANGE)), n_trials=TRIALS
        )
    return [(t.params["c"], t.params["g"], t.value) for t in study.trials]


def score_points(points):
    with threadpool_limits(1):
        return [(c, g, svm_score(c, g)) for c, g in points]


runs = {"grid": [score_points(grid_points())]}
runs["random"] = Parallel(n_jobs=N_JOBS)(delayed(score_points)(random_points(s)) for s in range(5))
runs["TPE"] = Parallel(n_jobs=N_JOBS)(delayed(tpe_points)(s) for s in range(5))
best_curves = {k: np.mean([np.maximum.accumulate([p[2] for p in r]) for r in v], axis=0) for k, v in runs.items()}
show(
    pl.DataFrame(
        [(k, float(c[4]), float(c[9]), float(c[24])) for k, c in best_curves.items()],
        schema=["method", "best after 5", "after 10", "after 25"],
        orient="row",
    )
)

In [ ]:
# plot e6_space
fg, axs = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.45), sharex=True, sharey=True)
fg.subplots_adjust(left=0.09, right=0.99, top=0.9, bottom=0.25, wspace=0.08)
for ax, (name, r) in zip(axs, runs.items()):
    pts = np.array(r[0])
    order = np.argsort(pts[:, 2])
    ax.scatter(pts[order, 0], pts[order, 1], c=pts[order, 2], cmap="viridis", vmin=0.4, vmax=0.9, s=11, linewidths=0)
    ax.set_title(name, fontsize=7.5, pad=2)
    ax.set_xlabel("$\\log_{10} C$")
axs[0].set_ylabel("$\\log_{10}\\gamma$")
plotlib.save(fg, "e6_space")

In [ ]:
# plot e6_search
fg, ax = plotlib.fig("full", 1.3)
for (name, c), col in zip(best_curves.items(), (GREY, BLUE, ORANGE)):
    ax.plot(np.arange(1, TRIALS + 1), c, color=col, label=name)
ax.set(xlabel="trials", ylabel="best CV MCC so far")
ax.legend(ncols=3, loc="lower right", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e6_search")

## Pruning

A trial that looks bad after few trees can stop. Boosting reports its validation score every 25 trees and Optuna's median pruner stops a trial that sits below the median of earlier trials at the same point.

In [ ]:
slide("Pruning: Stop a Bad Trial Early")
Xa, Xv, ya, yv = train_test_split(X_tr, y_tr, test_size=0.3, stratify=y_tr, random_state=SEED)


def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 20, log=True),
    }
    score = 0.0
    model = None
    for step in range(1, 13):
        model = XGBClassifier(n_estimators=25, random_state=SEED, n_jobs=4, **params).fit(Xa, ya, xgb_model=model)
        score = metrics.task_score("c", yv, model.predict(Xv))
        trial.report(score, step * 25)
        if trial.should_prune():
            trial.set_user_attr("trees", step * 25)
            raise optuna.TrialPruned
    trial.set_user_attr("trees", 300)
    return score


rows = []
studies = {}
for name, pruner in (
    ("no pruning", optuna.pruners.NopPruner()),
    ("median pruner", optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)),
):
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED), pruner=pruner)
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=30)
    trees = sum(t.user_attrs["trees"] for t in study.trials)
    rows.append(
        (name, study.best_value, trees, sum(t.state.name == "PRUNED" for t in study.trials), time.perf_counter() - t0)
    )
    studies[name] = study
pruning = pl.DataFrame(
    rows, schema=["search", "best validation MCC", "trees trained", "trials pruned", "seconds"], orient="row"
)
show(pruning)

In [ ]:
# plot e6_pruning
fg, ax = plotlib.fig("full", 1.3)
study = studies["median pruner"]
for t in study.trials:
    steps = sorted(t.intermediate_values)
    pruned = t.state.name == "PRUNED"
    ax.plot(steps, [t.intermediate_values[s] for s in steps], color=RED if pruned else BLUE, lw=0.6, alpha=0.7)
ax.plot([], [], color=BLUE, label="finished")
ax.plot([], [], color=RED, label="pruned")
ax.set(xlabel="trees", ylabel="validation MCC", ylim=(0.5, None))
ax.legend(ncols=2, loc="lower right", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e6_pruning")

## Which hyperparameter matters?

Optuna estimates the share of the score's variance that each hyperparameter explains (fANOVA): it tells which one to tune first.

In [ ]:
slide("Which Hyperparameter Matters?")
importance = optuna.importance.get_param_importances(studies["no pruning"])
show(pl.DataFrame({"hyperparameter": list(importance), "importance": list(importance.values())}))

In [ ]:
# plot e6_importance
fg, ax = plotlib.fig("full", 1.1)
names = list(importance)[::-1]
ax.barh(names, [importance[n] for n in names], color=ORANGE, height=0.6)
for i, n in enumerate(names):
    ax.text(importance[n] + 0.008, i, f"{importance[n]:.2f}", fontsize=6.2, va="center")
ax.set(xlabel="share of the variance of the score", xlim=(0, max(importance.values()) * 1.15))
ax.grid(axis="y", visible=False)
fg.subplots_adjust(left=0.24, right=0.98, bottom=0.32)
plotlib.save(fg, "e6_importance")

## The best validation score is optimistic

Picking the best of many trials selects on noise. On a small table the best inner CV score is above the score of the same choice on data it never saw. **Nested cross-validation:** the outer folds only score, the inner folds only tune. The sweep over the number of trials is the failure case: more trials, larger gap.

In [ ]:
slide("Nested Cross-Validation")
X_heart, y_heart = data.load_heart()


def nested_fold(tr, te, n_trials, seed):
    def objective(trial):
        model = make_pipeline(
            StandardScaler(), SVC(C=10 ** trial.suggest_float("c", -2, 3), gamma=10 ** trial.suggest_float("g", -5, 0))
        )
        return cross_val_score(
            model,
            X_heart[tr],
            y_heart[tr],
            cv=StratifiedKFold(3, shuffle=True, random_state=seed),
            scoring="matthews_corrcoef",
        ).mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    with threadpool_limits(1):
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_trials)
        p = study.best_params
        model = make_pipeline(StandardScaler(), SVC(C=10 ** p["c"], gamma=10 ** p["g"])).fit(X_heart[tr], y_heart[tr])
    return n_trials, study.best_value, metrics.task_score("c", y_heart[te], model.predict(X_heart[te]))


jobs = [
    (tr, te, n, s)
    for s in range(4)
    for n in (5, 25, 100)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=s).split(X_heart, y_heart)
]
nested = pl.DataFrame(
    Parallel(n_jobs=-1)(delayed(nested_fold)(*j) for j in jobs), schema=["trials", "inner", "outer"], orient="row"
)
nested_table = (
    nested.group_by("trials")
    .agg(pl.col("inner").mean(), pl.col("outer").mean(), pl.col("outer").std().alias("outer sd"))
    .sort("trials")
)
nested_table = nested_table.with_columns((pl.col("inner") - pl.col("outer")).alias("optimism"))
show(nested_table)

In [ ]:
# plot e6_nested
fg, ax = plotlib.fig("full", 1.25)
ax.plot(nested_table["trials"], nested_table["inner"], color=ORANGE, marker="o", ms=3, label="best inner CV score")
ax.plot(nested_table["trials"], nested_table["outer"], color=BLUE, marker="o", ms=3, label="outer score (honest)")
ax.set(xscale="log", xlabel="trials (log scale)", ylabel="MCC", xticks=[5, 25, 100], xticklabels=[5, 25, 100])
ax.legend(ncols=2, loc="lower center", bbox_to_anchor=(0.5, 1.0), fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e6_nested")

## Default against tuned

Five models on three real tables, five folds: scikit-learn defaults against 25 Optuna trials on the training folds.

In [ ]:
slide("Default Against Tuned")


TUNING_SETS = {"spambase": data.load_spam, "wine, good or not": data.wine_good, "breast cancer": data.load_breast}


def space(name, t):
    if name == "SVM":
        return make_pipeline(
            StandardScaler(),
            SVC(C=t.suggest_float("c", 1e-1, 300, log=True), gamma=t.suggest_float("g", 1e-4, 1, log=True)),
        )
    if name == "k-NN":
        return make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(
                n_neighbors=t.suggest_int("k", 1, 60), weights=t.suggest_categorical("w", ["uniform", "distance"])
            ),
        )
    if name == "random forest":
        return RandomForestClassifier(
            n_estimators=200,
            max_features=t.suggest_float("m", 0.1, 1.0),
            min_samples_leaf=t.suggest_int("leaf", 1, 10),
            random_state=SEED,
        )
    if name == "HistGB":
        return HistGradientBoostingClassifier(
            learning_rate=t.suggest_float("eta", 0.01, 0.3, log=True),
            max_leaf_nodes=t.suggest_int("leaves", 7, 63),
            min_samples_leaf=t.suggest_int("leaf", 5, 60),
            max_iter=300,
            random_state=SEED,
        )
    return XGBClassifier(
        n_estimators=300,
        learning_rate=t.suggest_float("eta", 0.01, 0.3, log=True),
        max_depth=t.suggest_int("depth", 2, 9),
        subsample=t.suggest_float("sub", 0.5, 1.0),
        colsample_bytree=t.suggest_float("col", 0.3, 1.0),
        n_jobs=1,
        random_state=SEED,
    )


DEFAULTS = {
    "SVM": make_pipeline(StandardScaler(), SVC()),
    "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "random forest": RandomForestClassifier(random_state=SEED),
    "HistGB": HistGradientBoostingClassifier(random_state=SEED),
    "XGBoost": XGBClassifier(n_jobs=1, random_state=SEED),
}


def default_tuned(dataset, name, tr, te):
    X, y = TUNING_SETS[dataset]()
    rows = np.random.default_rng(SEED).permutation(tr)[:1500]
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    with threadpool_limits(1), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(
            lambda t: cross_val_score(space(name, t), X[rows], y[rows], cv=cv, scoring="matthews_corrcoef").mean(),
            n_trials=25,
        )
        tuned = space(name, optuna.trial.FixedTrial(study.best_params)).fit(X[tr], y[tr])
        base = clone(DEFAULTS[name]).fit(X[tr], y[tr])
    return (
        dataset,
        name,
        metrics.task_score("c", y[te], base.predict(X[te])),
        metrics.task_score("c", y[te], tuned.predict(X[te])),
    )


CACHE = plotlib.ROOT / ".cache" / "07_tuning.csv"
if CACHE.exists():
    tuned_scores = pl.read_csv(CACHE)
else:
    jobs = []
    for d, load in TUNING_SETS.items():
        X, y = load()
        jobs += [
            (d, m, tr, te)
            for m in DEFAULTS
            for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y)
        ]
    tuned_scores = pl.DataFrame(
        Parallel(n_jobs=-1)(delayed(default_tuned)(*j) for j in jobs),
        schema=["dataset", "model", "default", "tuned"],
        orient="row",
    )
    CACHE.parent.mkdir(exist_ok=True)
    tuned_scores.write_csv(CACHE)
tuned_table = (
    tuned_scores.group_by("dataset", "model", maintain_order=True)
    .agg(pl.col("default").mean(), pl.col("tuned").mean())
    .with_columns((pl.col("tuned") - pl.col("default")).alias("gain"))
)
show(tuned_table)

In [ ]:
# plot e6_default
fg, axs = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.65), sharey=True)
fg.subplots_adjust(left=0.17, right=0.99, top=0.8, bottom=0.2, wspace=0.08)
models = list(DEFAULTS)[::-1]
for ax, d in zip(axs, TUNING_SETS):
    t = tuned_table.filter(pl.col("dataset") == d)
    vals = dict(zip(t["model"], zip(t["default"], t["tuned"])))
    y = np.arange(len(models))
    ax.barh(y + 0.19, [vals[m][1] for m in models], height=0.36, color=ORANGE, label="tuned")
    ax.barh(y - 0.19, [vals[m][0] for m in models], height=0.36, color=GREY, label="default")
    low = min(min(v) for v in vals.values())
    ax.set_xlim(low * 0.92, max(max(v) for v in vals.values()) * 1.02)
    ax.set_title(d, fontsize=6.8, pad=2)
    ax.set_xlabel("MCC", fontsize=6.3)
    ax.tick_params(labelsize=5.8)
    ax.grid(axis="y", visible=False)
axs[0].set_yticks(np.arange(len(models)), models)
axs[1].legend(ncols=2, loc="lower center", bbox_to_anchor=(0.5, 1.16), fontsize=6, handlelength=1)
plotlib.save(fg, "e6_default")